In [2]:
! pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 81.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 66.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 105.1 MB/s eta 0:00:0000:0100:01


In [2]:
import pennylane as qml
from pennylane import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import joblib
import os
import matplotlib.pyplot as plt

In [3]:
def preprocess_weather_data(csv_path, seq_len=336, pred_len=96):
    # 1. Load Data
    df = pd.read_csv(csv_path)
    df['date'] = pd.to_datetime(df['date'])
    df.set_index('date', inplace=True)
    
    # --- NEW: Handle Missing Values ---
    # Check for NaNs
    if df.isnull().values.any():
        print(f"Found {df.isnull().sum().sum()} missing values. Filling with forward fill...")
        df.fillna(method='ffill', inplace=True) # Fill gaps with previous value
        df.fillna(0, inplace=True) # Fill remaining (start of file) with 0
    
    # Ensure all data is float type
    data = df.values.astype(np.float32)
    
    print(f"Data Shape: {data.shape}")

    # 2. Split Data (70% Train, 10% Val, 20% Test)
    n_train = int(len(data) * 0.7)
    n_test = int(len(data) * 0.2)
    n_val = len(data) - n_train - n_test

    train_data = data[:n_train]
    val_data = data[n_train : n_train + n_val]
    test_data = data[n_train + n_val:]

    # 3. Normalization
    scaler = StandardScaler()
    scaler.fit(train_data)

    train_scaled = scaler.transform(train_data)
    val_scaled = scaler.transform(val_data)
    test_scaled = scaler.transform(test_data)

    # 4. Create Sliding Windows
    def create_sequences(dataset, seq_len, pred_len):
        X, Y = [], []
        for i in range(len(dataset) - seq_len - pred_len + 1):
            s_begin = i
            s_end = s_begin + seq_len
            r_begin = s_end
            r_end = r_begin + pred_len

            X.append(dataset[s_begin:s_end])
            Y.append(dataset[r_begin:r_end])
            
        return np.array(X), np.array(Y)

    x_train, y_train = create_sequences(train_scaled, seq_len, pred_len)
    x_val, y_val = create_sequences(val_scaled, seq_len, pred_len)
    x_test, y_test = create_sequences(test_scaled, seq_len, pred_len)
    
    # --- FINAL CHECK ---
    if np.isnan(x_train).any():
        raise ValueError("Data still contains NaNs after preprocessing!")

    print(f"Train Sequences: {x_train.shape}")
    
    return {
        'train': (x_train, y_train),
        'val': (x_val, y_val),
        'test': (x_test, y_test),
        'scaler': scaler
    }

In [4]:
def save_checkpoint(model, scaler, filepath="qultsf_checkpoint.pth"):
    """
    Saves the model state, architecture configs, and the data scaler.
    """
    # 1. Create a dictionary with everything needed to restore the model
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'configs': {
            'seq_len': model.seq_len,
            'pred_len': model.pred_len,
            # 'num_qubits': model.num_qubits,
            'patch_len': model.patch_len,  # Added these
            'stride': model.stride,        
            'num_layers': model.num_layers,
            'QML_device': model.QML_device
        }
    }
    
    # 2. Save the PyTorch/PennyLane part
    torch.save(checkpoint, filepath)
    print(f"Model weights and config saved to {filepath}")
    
    # 3. Save the Scaler (StandardScaler) separately
    scaler_path = filepath.replace(".pth", "_scaler.pkl")
    joblib.dump(scaler, scaler_path)
    print(f"Data Scaler saved to {scaler_path}")

class SimpleNamespace:
    """A simple helper class to convert a dictionary to an object like 'configs'."""
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

def load_checkpoint(filepath="qultsf_checkpoint.pth", device='cpu'):
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"No model found at {filepath}")

    # 1. Load the dictionary
    checkpoint = torch.load(filepath, map_location=device)
    config_dict = checkpoint['configs']
    
    print(f"Loading model with configs: {config_dict}")

    # 2. Reconstruct the Config object
    # This turns the dictionary back into an object so 'configs.seq_len' works
    configs = SimpleNamespace(**config_dict)

    # 3. Initialize Model (Passing the single config object)
    model = Model(configs)
    
    # 4. Load weights
    model.load_state_dict(checkpoint['model_state_dict'])
    model.to(device)
    model.eval()
    
    # 5. Load Scaler
    scaler_path = filepath.replace(".pth", "_scaler.pkl")
    scaler = joblib.load(scaler_path)
    
    print("Model and Scaler loaded successfully.")
    return model, scaler

# Original Model

In [5]:
class Model(nn.Module):
    def __init__(self, configs):
        super(Model, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.num_qubits = configs.num_qubits
        self.QML_device = configs.QML_device
        self.num_layers = configs.num_layers  # This is 'K' in the paper

        self.hybrid_qml_model = Hybrid_QML_Model(
            lookback_window_size=self.seq_len,
            forecast_window_size=self.pred_len,
            num_qubits=self.num_qubits,
            QML_device=self.QML_device,
            num_layers=self.num_layers
        )

    def forward(self, x):
        # x shape comes in as: [Batch, Seq_Len, Variates]
        # Example: [16, 336, 21]
        
        # Pass to the hybrid model
        x = self.hybrid_qml_model(x)
        
        # Returns: [Batch, Pred_Len, Variates]
        return x 

    def summary(self):
        """Prints a summary of the model architecture and parameters."""
        print("\n" + "="*80)
        print(f"{'Layer / Parameter Name':<40} | {'Shape':<20} | {'Params'}")
        print("-" * 80)
        
        total_params = 0
        trainable_params = 0
        
        for name, param in self.named_parameters():
            if not param.requires_grad: 
                continue
            
            # Get parameter count
            param_count = param.numel()
            total_params += param_count
            trainable_params += param_count
            
            # Format shape as string
            shape_str = str(list(param.shape))
            
            print(f"{name:<40} | {shape_str:<20} | {param_count:,}")
            
        print("-" * 80)
        print(f"Total Parameters:     {total_params:,}")
        print(f"Trainable Parameters: {trainable_params:,}")
        print("="*80 + "\n")

class Hybrid_QML_Model(nn.Module):
    def __init__(self, lookback_window_size, forecast_window_size, num_qubits, QML_device, num_layers):
        super(Hybrid_QML_Model, self).__init__()
        self.forecast_window_size = forecast_window_size
        self.num_qubits = num_qubits
        
        # Initialize Quantum Device
        # Use 'lightning.qubit' for CPU speed or 'lightning.gpu' if you have CuQuantum
        self.dev = qml.device(QML_device, wires=num_qubits)

        @qml.qnode(self.dev, interface='torch', diff_method="best")
        def quantum_function(inputs, weights):
            # 1. Amplitude Embedding (Maps 2^N features to quantum state)
            qml.AmplitudeEmbedding(features=inputs, wires=range(num_qubits), normalize=True)
            
            # 2. Variational Quantum Circuit (Ansatz)
            # Paper uses Hardware Efficient Ansatz (Rotations + CNOTs)
            # StronglyEntanglingLayers is the standard implementation of this
            qml.StronglyEntanglingLayers(weights=weights, wires=range(num_qubits))
            
            # 3. Measurement
            return [qml.expval(qml.PauliZ(i)) for i in range(num_qubits)]

        self._quantum_circuit = quantum_function
        
        # Shape: (Layers, Qubits, 3 parameters per Rot gate) -> Matches paper's "3NK" parameters
        q_weights_shape = {'weights': (num_layers, num_qubits, 3)}
        
        # --- Layers ---
        # 1. Input Linear: Maps Sequence Length (L) -> 2^N
        self.input_classical_layer = torch.nn.Linear(lookback_window_size, 2 ** num_qubits)
        
        # 2. Quantum Layer
        self.hidden_quantum_layer = qml.qnn.TorchLayer(self._quantum_circuit, q_weights_shape)
        
        # 3. Output Linear: Maps N Qubits -> Prediction Length (T)
        self.output_classical_layer = torch.nn.Linear(num_qubits, forecast_window_size)

    def forward(self, x):
        # x shape: [Batch, Seq_Len, Variates] (e.g., [16, 336, 21])
        B, L, M = x.shape
        
        # --- 1. Channel Independence Reshaping ---
        # We need to treat every variate as an independent sample.
        # Permute to [Batch, Variates, Seq_Len] -> [16, 21, 336]
        x = x.permute(0, 2, 1)
        
        # Flatten Batch and Variates -> [Batch * Variates, Seq_Len] -> [336, 336]
        # This creates the "Channel Independence" effect
        x = x.reshape(B * M, L)
        
        # --- 2. Input Linear Layer ---
        # Input: [336, 336] -> Output: [336, 1024] (if 10 qubits)
        x = self.input_classical_layer(x)
        
        # --- 3. Quantum Layer ---
        # Input: [336, 1024] -> Output: [336, 10] (10 qubits measured)
        # Note: AmplitudeEmbedding automatically normalizes the input vector if normalize=True
        x = self.hidden_quantum_layer(x)
        
        # --- 4. Output Linear Layer ---
        # Input: [336, 10] -> Output: [336, 96] (Prediction Length)
        x = self.output_classical_layer(x)
        
        # --- 5. Reshape back to Time Series Format ---
        # Current: [Batch * Variates, Pred_Len] -> [336, 96]
        # Reshape to [Batch, Variates, Pred_Len] -> [16, 21, 96]
        x = x.reshape(B, M, -1)
        
        # Permute back to [Batch, Pred_Len, Variates] -> [16, 96, 21]
        x = x.permute(0, 2, 1)
        
        return x

## Experiment 1: Skip Connections

In [6]:
class Model(nn.Module):
    def __init__(self, configs):
        super(Model, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.num_qubits = configs.num_qubits
        self.QML_device = configs.QML_device
        self.num_layers = configs.num_layers  # This is 'K' in the paper

        self.hybrid_qml_model = Hybrid_QML_Model(
            lookback_window_size=self.seq_len,
            forecast_window_size=self.pred_len,
            num_qubits=self.num_qubits,
            QML_device=self.QML_device,
            num_layers=self.num_layers
        )

    def forward(self, x):
        # x shape comes in as: [Batch, Seq_Len, Variates]
        # Example: [16, 336, 21]
        
        # Pass to the hybrid model
        x = self.hybrid_qml_model(x)
        
        # Returns: [Batch, Pred_Len, Variates]
        return x 

    def summary(self):
        """Prints a summary of the model architecture and parameters."""
        print("\n" + "="*80)
        print(f"{'Layer / Parameter Name':<40} | {'Shape':<20} | {'Params'}")
        print("-" * 80)
        
        total_params = 0
        trainable_params = 0
        
        for name, param in self.named_parameters():
            if not param.requires_grad: 
                continue
            
            # Get parameter count
            param_count = param.numel()
            total_params += param_count
            trainable_params += param_count
            
            # Format shape as string
            shape_str = str(list(param.shape))
            
            print(f"{name:<40} | {shape_str:<20} | {param_count:,}")
            
        print("-" * 80)
        print(f"Total Parameters:     {total_params:,}")
        print(f"Trainable Parameters: {trainable_params:,}")
        print("="*80 + "\n")

class Hybrid_QML_Model(nn.Module):
    def __init__(self, lookback_window_size, forecast_window_size, num_qubits, QML_device, num_layers):
        super(Hybrid_QML_Model, self).__init__()
        self.lookback_window_size = lookback_window_size
        self.forecast_window_size = forecast_window_size
        self.num_qubits = num_qubits
        
        # --- QUANTUM BRANCH ---
        self.dev = qml.device(QML_device, wires=num_qubits)

        @qml.qnode(self.dev, interface='torch', diff_method="best")
        def quantum_function(inputs, weights):
            qml.AmplitudeEmbedding(features=inputs, wires=range(num_qubits), normalize=True)
            qml.StronglyEntanglingLayers(weights=weights, wires=range(num_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(num_qubits)]

        self._quantum_circuit = quantum_function
        q_weights_shape = {'weights': (num_layers, num_qubits, 3)}
        
        self.input_classical_layer = torch.nn.Linear(lookback_window_size, 2 ** num_qubits)
        self.hidden_quantum_layer = qml.qnn.TorchLayer(self._quantum_circuit, q_weights_shape)
        self.output_classical_layer = torch.nn.Linear(num_qubits, forecast_window_size)

        # --- IMPROVEMENT: CLASSICAL RESIDUAL CONNECTION ---
        # A simple linear layer that bypasses the quantum circuit entirely.
        # It maps Input (336) directly to Output (96).
        self.skip_connection = torch.nn.Linear(lookback_window_size, forecast_window_size)
        
        # Optional: Initialize skip connection to be dominant at start
        # This helps the model start with a reasonable linear guess
        torch.nn.init.xavier_uniform_(self.skip_connection.weight)


    def forward(self, x):
        # 1. Handle Dimensions (Channel Independence)
        # We need [Total_Samples, Seq_Len]
        B = x.shape[0]
        
        if x.shape[-1] == self.lookback_window_size:
            x_flat = x.reshape(-1, self.lookback_window_size) # [B*M, 336]
        elif x.shape[1] == self.lookback_window_size:
             x_flat = x.permute(0, 2, 1).reshape(-1, self.lookback_window_size)
        else:
            raise ValueError(f"Shape mismatch: {x.shape}")

        # 2. Quantum Path (The "Complex" learner)
        # Input -> Linear -> Quantum -> Linear -> Output
        q_out = self.input_classical_layer(x_flat)
        q_out = self.hidden_quantum_layer(q_out)
        q_out = self.output_classical_layer(q_out)
        
        # 3. Classical Skip Path (The "Trend" learner)
        # Input -> Linear -> Output
        c_out = self.skip_connection(x_flat)
        
        # 4. Combine (Add them together)
        final_output = q_out + c_out
        
        # 5. Reshape back to [Batch, Pred_Len, Variates]
        M = final_output.shape[0] // B
        final_output = final_output.reshape(B, M, -1).permute(0, 2, 1)
        
        return final_output

## Experiment 2: Q-D Linear

In [7]:
# ==========================================
# 1. Decomposition Block (Moving Average)
# ==========================================
class MovingAverage(nn.Module):
    """
    Computes the Trend using a simple moving average kernel.
    The 'Seasonal' part is then (Input - Trend).
    """
    def __init__(self, kernel_size, stride):
        super(MovingAverage, self).__init__()
        self.kernel_size = kernel_size
        self.avg = nn.AvgPool1d(kernel_size=kernel_size, stride=stride, padding=0)

    def forward(self, x):
        # x shape: [Batch, Length]
        # We need to pad the ends so the Output Length == Input Length
        # Padding strategy: Replicate the first and last elements
        front = x[:, 0:1].repeat(1, (self.kernel_size - 1) // 2)
        end = x[:, -1:].repeat(1, (self.kernel_size - 1) // 2)
        
        x_padded = torch.cat([front, x, end], dim=1)
        
        # Calculate Trend
        # AvgPool expects [Batch, Channels, Length], so we unsqueeze dim 1
        trend = self.avg(x_padded.unsqueeze(1)).squeeze(1)
        
        # Calculate Seasonal (The wiggles)
        seasonal = x - trend
        return seasonal, trend


# ==========================================
# 2. Quantum Seasonal Block
# ==========================================
class Seasonal_Quantum_Block(nn.Module):
    """
    This block handles ONLY the seasonal (fluctuating) component.
    It uses Amplitude Embedding -> VQC -> Measurement.
    """
    def __init__(self, seq_len, pred_len, num_qubits, num_layers, QML_device):
        super().__init__()
        
        # Quantum Device
        self.dev = qml.device(QML_device, wires=num_qubits)

        # Define Circuit
        @qml.qnode(self.dev, interface='torch', diff_method="best")
        def quantum_function(inputs, weights):
            # Embed the Seasonal data (which is centered around 0 now!)
            qml.AmplitudeEmbedding(features=inputs, wires=range(num_qubits), normalize=True)
            # Entangling Layers to learn the periodic patterns
            qml.StronglyEntanglingLayers(weights=weights, wires=range(num_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(num_qubits)]

        self._quantum_circuit = quantum_function
        q_weights_shape = {'weights': (num_layers, num_qubits, 3)}
        
        # Layers
        # 1. Compress 336 seasonal steps -> 2^N quantum state
        self.input_linear = nn.Linear(seq_len, 2 ** num_qubits)
        
        # 2. The Quantum Circuit
        self.q_layer = qml.qnn.TorchLayer(self._quantum_circuit, q_weights_shape)
        
        # 3. Map N Qubits -> 96 seasonal prediction steps
        self.output_linear = nn.Linear(num_qubits, pred_len)

    def forward(self, x):
        # x: [Batch, Seq_Len] (Seasonal component)
        x = self.input_linear(x)
        x = self.q_layer(x)
        x = self.output_linear(x)
        return x


# ==========================================
# 3. Main Model (Decomposition-QuLTSF)
# ==========================================
class Model(nn.Module):
    def __init__(self, configs):
        super(Model, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.num_qubits = configs.num_qubits
        self.num_layers = configs.num_layers
        self.QML_device = configs.QML_device
        # Decomposition Kernel Size (Usually odd number, e.g., 25)
        kernel_size = 25
        self.decomposition = MovingAverage(kernel_size, stride=1)
        
        # --- BRANCH 1: CLASSICAL TREND ---
        # Handles the "Linear" global warming style trends
        self.trend_model = nn.Linear(configs.seq_len, configs.pred_len)
        
        # --- BRANCH 2: QUANTUM SEASONAL ---
        # Handles the complex day/night cycles
        self.seasonal_model = Seasonal_Quantum_Block(
            configs.seq_len, 
            configs.pred_len, 
            configs.num_qubits, 
            configs.num_layers,
            configs.QML_device
        )

    def forward(self, x):
        # x shape: [Batch, Seq_Len, Variates] -> e.g., [16, 336, 21]
        B, L, M = x.shape
        
        # 1. Channel Independence Reshaping
        # Treat every variable as an independent sequence
        # [Batch, Variates, Seq_Len] -> [Batch * Variates, Seq_Len]
        x_flat = x.permute(0, 2, 1).reshape(B * M, L)
        
        # 2. Decompose into Trend and Seasonal
        # seasonal: [B*M, 336] (Mean ~ 0)
        # trend:    [B*M, 336] (Smoothed line)
        seasonal_init, trend_init = self.decomposition(x_flat)
        
        # 3. Process Trend (Classical)
        trend_output = self.trend_model(trend_init)
        
        # 4. Process Seasonal (Quantum)
        seasonal_output = self.seasonal_model(seasonal_init)
        
        # 5. Add them back together
        final_output = seasonal_output + trend_output
        
        # 6. Reshape back to [Batch, Pred_Len, Variates]
        final_output = final_output.reshape(B, M, -1).permute(0, 2, 1)
        
        return final_output

    def summary(self):
        print("\n" + "="*60)
        print("Model: Decomposition-QuLTSF (Q-DLinear)")
        print("="*60)
        total_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"Trend Path (Classical):   Maintains low-freq drift")
        print(f"Seasonal Path (Quantum):  Learns high-freq cycles")
        print(f"Total Parameters:         {total_params:,}")
        print("="*60 + "\n")

## Experiment 3: Patch-QuLTSF

In [8]:
class RevIN(nn.Module):
    def __init__(self, num_features: int, eps=1e-5, affine=True):
        """
        Reversible Instance Normalization
        :param num_features: the number of features or channels
        :param eps: a value added for numerical stability
        :param affine: if True, RevIN has learnable affine parameters
        """
        super(RevIN, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        if self.affine:
            self._init_params()

    def _init_params(self):
        # initialize RevIN params: (C,)
        self.affine_weight = nn.Parameter(torch.ones(self.num_features))
        self.affine_bias = nn.Parameter(torch.zeros(self.num_features))

    def _get_statistics(self, x):
        dim2reduce = tuple(range(1, x.ndim-1))
        self.mean = torch.mean(x, dim=dim2reduce, keepdim=True).detach()
        self.stdev = torch.sqrt(torch.var(x, dim=dim2reduce, keepdim=True, unbiased=False) + self.eps).detach()

    def _normalize(self, x):
        x = x - self.mean
        x = x / self.stdev
        if self.affine:
            x = x * self.affine_weight
            x = x + self.affine_bias
        return x

    def _denormalize(self, x):
        if self.affine:
            x = x - self.affine_bias
            x = x / (self.affine_weight + self.eps*self.eps)
        x = x * self.stdev
        x = x + self.mean
        return x

    def forward(self, x, mode:str):
        if mode == 'norm':
            self._get_statistics(x)
            x = self._normalize(x)
        elif mode == 'denorm':
            x = self._denormalize(x)
        else: raise NotImplementedError
        return x

In [9]:
class Model(nn.Module):
    def __init__(self, configs):
        super(Model, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.num_layers = configs.num_layers
        self.QML_device = configs.QML_device
        self.patch_len = configs.patch_len
        self.stride = 16
        # 1. RevIN for normalization
        self.num_variates = 21 
        self.revin = RevIN(self.num_variates)

        # 2. Patch Model (Pure Quantum logic)
        self.patch_qml_model = Patch_QML_Model(
            seq_len=self.seq_len,
            pred_len=self.pred_len,
            patch_len=self.patch_len,
            stride=16,
            num_layers=self.num_layers,
            QML_device=self.QML_device
        )

    def forward(self, x):
        # x shape: [Batch, Seq_Len, Variates]
        x = self.revin(x, 'norm')
        x = self.patch_qml_model(x)
        x = self.revin(x, 'denorm')
        return x 

    def summary(self):
       
        """Prints a summary of the model architecture, patching logic, and parameters."""
        print("\n" + "="*80)
        print(f"{'Patch-QuLTSF Model Summary':^80}")
        print("="*80)
        
#         # 1. Print Logic Configuration
        print(f"Configuration:")
        print(f"  Input Sequence Length:   {self.seq_len}")
        print(f"  Prediction Length:       {self.pred_len}")
        print(f"  Patch Length:            {self.patch_len}")
        print(f"  Stride:                  {self.stride}")
        print(f"  Total Patches:           {self.patch_qml_model.num_patches}")
        print(f"  Qubits per Patch:        {self.patch_qml_model.num_qubits}")
        print("-" * 80)
        
        # 2. Print Parameter Table
        print(f"{'Layer / Parameter Name':<40} | {'Shape':<20} | {'Params'}")
        print("-" * 80)
        
        total_params = 0
        trainable_params = 0
        
        for name, param in self.named_parameters():
            if not param.requires_grad: 
                continue
            
            param_count = param.numel()
            total_params += param_count
            trainable_params += param_count
            
            shape_str = str(list(param.shape))
            print(f"{name:<40} | {shape_str:<20} | {param_count:,}")
            
        print("-" * 80)
        print(f"Total Parameters:     {total_params:,}")
        print(f"Trainable Parameters: {trainable_params:,}")
        print("="*80 + "\n")

class Patch_QML_Model(nn.Module):
    def __init__(self, seq_len, pred_len, patch_len, stride, num_layers, QML_device):
        super(Patch_QML_Model, self).__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.patch_len = patch_len
        self.stride = stride
        
        self.num_patches = (self.seq_len - self.patch_len) // self.stride + 1
        self.num_qubits = 4 
        
        # --- QUANTUM BRANCH ---
        self.dev = qml.device(QML_device, wires=self.num_qubits)

        @qml.qnode(self.dev, interface='torch', diff_method="best")
        def quantum_function(inputs, weights):
            qml.AmplitudeEmbedding(features=inputs, wires=range(self.num_qubits), normalize=True)
            qml.StronglyEntanglingLayers(weights=weights, wires=range(self.num_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(self.num_qubits)]

        self._quantum_circuit = quantum_function
        q_weights_shape = {'weights': (num_layers, self.num_qubits, 3)}
        
        # Quantum Encoder
        self.patch_encoder = qml.qnn.TorchLayer(self._quantum_circuit, q_weights_shape)
        
        # Output Projection (Head)
        flattened_dim = self.num_patches * self.num_qubits
        self.head = nn.Linear(flattened_dim, self.pred_len)

        # NOTE: self.skip_connection has been REMOVED

    def forward(self, x):
        B = x.shape[0]
        
        # 1. Channel Independence & Flattening
        if x.shape[-1] == self.seq_len:
             x_flat = x.reshape(-1, self.seq_len)
        elif x.shape[1] == self.seq_len:
             x_flat = x.permute(0, 2, 1).reshape(-1, self.seq_len)
        else:
            raise ValueError(f"Shape mismatch: {x.shape}")
            
        # 2. Patching (Unfold)
        x_patches = x_flat.unfold(dimension=1, size=self.patch_len, step=self.stride)
        
        # 3. Stack patches for parallel quantum processing
        x_patches_stacked = x_patches.reshape(-1, self.patch_len)
        
        # Stability fix for AmplitudeEmbedding
        x_patches_stacked = x_patches_stacked + 1e-6 
        
        # 4. Quantum Encoding
        # [Samples*Patches, 16] -> [Samples*Patches, 4]
        encoded_patches = self.patch_encoder(x_patches_stacked)
        
        # 5. Reshape and Flatten
        encoded_patches = encoded_patches.reshape(x_flat.shape[0], self.num_patches, self.num_qubits)
        features = encoded_patches.reshape(x_flat.shape[0], -1)
        
        # 6. Final Projection (Only Path)
        # The model now depends entirely on these weights to generate the forecast
        out = self.head(features)
        
        # 7. Restore Dimensions [Batch, Pred_Len, Variates]
        M = out.shape[0] // B
        out = out.reshape(B, M, -1).permute(0, 2, 1)
        
        return out

## Experiment 4: Patch QuLTSF + Residual Connection

In [10]:
class Model(nn.Module):
    def __init__(self, configs):
        super(Model, self).__init__()
        self.seq_len = configs.seq_len
        self.pred_len = configs.pred_len
        self.num_layers = configs.num_layers
        self.QML_device = configs.QML_device
        
        # Patching hyperparameters (Can be moved to configs if desired)
        self.patch_len = 16  # P
        self.stride = 16     # S (Non-overlapping)

        # Initialize the specific Patch-based implementation
        self.patch_qml_model = Patch_QML_Model(
            seq_len=self.seq_len,
            pred_len=self.pred_len,
            patch_len=self.patch_len,
            stride=self.stride,
            num_layers=self.num_layers,
            QML_device=self.QML_device
        )

    def forward(self, x):
        # x shape: [Batch, Seq_Len, Variates]
        x = self.patch_qml_model(x)
        return x 

    def summary(self):
        """Prints a summary of the model architecture, patching logic, and parameters."""
        print("\n" + "="*80)
        print(f"{'Patch-QuLTSF Model Summary':^80}")
        print("="*80)
        
        # 1. Print Logic Configuration
        print(f"Configuration:")
        print(f"  Input Sequence Length:   {self.seq_len}")
        print(f"  Prediction Length:       {self.pred_len}")
        print(f"  Patch Length:            {self.patch_len}")
        print(f"  Stride:                  {self.stride}")
        print(f"  Total Patches:           {self.patch_qml_model.num_patches}")
        print(f"  Qubits per Patch:        {self.patch_qml_model.num_qubits}")
        print("-" * 80)
        
        # 2. Print Parameter Table
        print(f"{'Layer / Parameter Name':<40} | {'Shape':<20} | {'Params'}")
        print("-" * 80)
        
        total_params = 0
        trainable_params = 0
        
        for name, param in self.named_parameters():
            if not param.requires_grad: 
                continue
            
            param_count = param.numel()
            total_params += param_count
            trainable_params += param_count
            
            shape_str = str(list(param.shape))
            print(f"{name:<40} | {shape_str:<20} | {param_count:,}")
            
        print("-" * 80)
        print(f"Total Parameters:     {total_params:,}")
        print(f"Trainable Parameters: {trainable_params:,}")
        print("="*80 + "\n")


class Patch_QML_Model(nn.Module):
    def __init__(self, seq_len, pred_len, patch_len, stride, num_layers, QML_device):
        super(Patch_QML_Model, self).__init__()
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.patch_len = patch_len
        self.stride = stride
        
        # 1. Calculate Number of Patches
        # Formula: N = (L - P) / S + 1
        self.num_patches = (self.seq_len - self.patch_len) // self.stride + 1
        
        # 2. Quantum Config
        # We match qubits to patch_len (2^4 = 16) for dense embedding
        self.num_qubits = 4 
        
        # --- QUANTUM BRANCH (Shared Encoder) ---
        self.dev = qml.device(QML_device, wires=self.num_qubits)

        @qml.qnode(self.dev, interface='torch', diff_method="best")
        def quantum_function(inputs, weights):
            # Embed 16 floats into 4 qubits
            qml.AmplitudeEmbedding(features=inputs, wires=range(self.num_qubits), normalize=True)
            # Variational Layers
            qml.StronglyEntanglingLayers(weights=weights, wires=range(self.num_qubits))
            # Measure all 4 qubits
            return [qml.expval(qml.PauliZ(i)) for i in range(self.num_qubits)]

        self._quantum_circuit = quantum_function
        
        # Weights: [Layers, Qubits, 3 params]
        q_weights_shape = {'weights': (num_layers, self.num_qubits, 3)}
        
        # The Quantum Encoder Layer
        self.patch_encoder = qml.qnn.TorchLayer(self._quantum_circuit, q_weights_shape)
        
        # --- CLASSICAL PROJECTION ---
        # Input to head = Num_Patches * Num_Qubits_Output (e.g., 21 * 4 = 84)
        flattened_dim = self.num_patches * self.num_qubits
        self.head = nn.Linear(flattened_dim, self.pred_len)
        
        # --- CLASSICAL RESIDUAL CONNECTION ---
        # Maps full sequence to prediction directly (Global Skip)
        self.skip_connection = nn.Linear(self.seq_len, self.pred_len)
        
        # Initialize skip to be dominant for stability
        torch.nn.init.xavier_uniform_(self.skip_connection.weight)

    def forward(self, x):
        # x shape: [Batch, Seq_Len, Variates] -> [16, 336, 21]
        B = x.shape[0]
        
        # 1. Channel Independence & Flattening
        # Check input orientation and flatten to [Batch * Variates, Seq_Len]
        if x.shape[-1] == self.seq_len:
            # Already [Batch, Variates, Seq_Len] -> [16, 21, 336]
             x_flat = x.reshape(-1, self.seq_len)
        elif x.shape[1] == self.seq_len:
            # Is [Batch, Seq_Len, Variates] -> [16, 336, 21]
             x_flat = x.permute(0, 2, 1).reshape(-1, self.seq_len)
        else:
            raise ValueError(f"Shape mismatch: {x.shape} does not match seq_len {self.seq_len}")
            
        # Current x_flat: [Total_Samples, 336]
        
        # 2. Patching (Unfold)
        # Output: [Total_Samples, Num_Patches, Patch_Len]
        # Shape: [336, 21, 16]
        x_patches = x_flat.unfold(dimension=1, size=self.patch_len, step=self.stride)
        
        # 3. Stack for Parallel Quantum Processing
        # We need to treat every patch as a separate sample for the QNode
        # [Total_Samples * Num_Patches, Patch_Len] -> [7056, 16]
        x_patches_stacked = x_patches.reshape(-1, self.patch_len)
        
        # 4. Quantum Encoding
        # [7056, 16] -> [7056, 4]
        encoded_patches = self.patch_encoder(x_patches_stacked)
        
        # 5. Reshape and Flatten Features
        # Back to [Total_Samples, Num_Patches, Num_Qubits]
        encoded_patches = encoded_patches.reshape(x_flat.shape[0], self.num_patches, self.num_qubits)
        
        # Flatten patches: [Total_Samples, Num_Patches * Num_Qubits]
        # Shape: [336, 21*4] -> [336, 84]
        features = encoded_patches.reshape(x_flat.shape[0], -1)
        
        # 6. Final Projection
        # [336, 84] -> [336, 96]
        q_out = self.head(features)
        
        # 7. Residual Skip Connection
        # [336, 336] -> [336, 96]
        c_out = self.skip_connection(x_flat)
        
        # 8. Combine
        final_output = q_out + c_out
        
        # 9. Restore Dimensions [Batch, Pred_Len, Variates]
        # Total_Samples = Batch * Variates. 
        # We divide by Batch (B) to get Variates (M)
        M = final_output.shape[0] // B
        final_output = final_output.reshape(B, M, -1).permute(0, 2, 1)
        
        return final_output

## Training Loop

In [11]:
# Wrapper for the dataset
class WeatherDataset(Dataset):
    def __init__(self, x_data, y_data):
        self.x = torch.FloatTensor(x_data)
        self.y = torch.FloatTensor(y_data)
    def __len__(self):
        return len(self.x)
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [12]:
def train_qultsf(model, data_dict, seq_len, pred_len, batch_size, epochs, lr, device):
    # 1. Setup DataLoaders
    train_set = WeatherDataset(data_dict['train'][0], data_dict['train'][1])
    val_set   = WeatherDataset(data_dict['val'][0],   data_dict['val'][1])
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, drop_last=True)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.7, patience=1)

    criterion = nn.MSELoss() 
    mae_criterion = nn.L1Loss() 

    print(f"Starting training on {device}...")
    
    for epoch in range(epochs):
        model.train()
        total_mse = 0
        
        for i, (batch_x, batch_y) in enumerate(train_loader):
            batch_x = batch_x.to(device) # Shape: [16, 336, 21]
            batch_y = batch_y.to(device) # Shape: [16, 96, 21]

            optimizer.zero_grad()
            
            # --- FIX IS HERE ---
            # Do NOT reshape batch_x manually. 
            # Pass the 3D tensor directly. The model handles the internals.
            outputs = model(batch_x) 
            
            # outputs shape: [16, 96, 21]
            # batch_y shape: [16, 96, 21]
            # MSELoss can handle 3D tensors directly
            loss = criterion(outputs, batch_y)
            
            loss.backward()
            optimizer.step()
            
            total_mse += loss.item()

        # Validation Phase
        model.eval()
        val_mse = 0
        val_mae = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                
                outputs = model(batch_x)
                
                val_mse += criterion(outputs, batch_y).item()
                val_mae += mae_criterion(outputs, batch_y).item()

        avg_train_loss = total_mse / len(train_loader)
        avg_val_mse = val_mse / len(val_loader)
        avg_val_mae = val_mae / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']

        scheduler.step(avg_val_mse)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train MSE: {avg_train_loss:.3f} | "
              f"Val MSE: {avg_val_mse:.3f} | "
              f"Val MAE: {avg_val_mae:.3f} | "
              f"Current LR: {current_lr:.5f}")
              
    return model

In [13]:
def train_patched_qultsf(model, data_dict, seq_len, pred_len, batch_size, epochs, lr, device):
    # 1. Setup DataLoaders
    train_set = WeatherDataset(data_dict['train'][0], data_dict['train'][1])
    val_set   = WeatherDataset(data_dict['val'][0],   data_dict['val'][1])
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False, drop_last=True)

    # ============================================================
    # IMPROVEMENT: SEPARATE PARAMETER GROUPS
    # ============================================================
    
    # 1. Identify Quantum Parameters (The VQC weights)
    # We access the sub-module we defined in 'Patch_QML_Model'
    quantum_params = list(model.patch_qml_model.patch_encoder.parameters())
    
    # 2. Identify Classical Parameters (Skip connection, Head, etc.)
    # We calculate the IDs of quantum params to exclude them
    quantum_ids = list(map(id, quantum_params))
    classical_params = filter(lambda p: id(p) not in quantum_ids, model.parameters())

    # 3. Define Optimizer with different Learning Rates
    optimizer = optim.Adam([
        {'params': classical_params, 'lr': 0.0005},   # Fast LR for Linear Layers
        {'params': quantum_params,   'lr': 0.00005}   # Slow/Stable LR for Quantum Circuit
    ])
    
    # Scheduler monitors the *Quantum* progress mostly
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    criterion = nn.MSELoss() 
    mae_criterion = nn.L1Loss() 

    print(f"Starting training on {device}...")
    print(f"Classical LR: 0.001 | Quantum LR: 0.0001")
    
    for epoch in range(epochs):
        model.train()
        total_mse = 0
        
        for i, (batch_x, batch_y) in enumerate(train_loader):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_x) 
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_mse += loss.item()

        # Validation Phase
        model.eval()
        val_mse = 0
        val_mae = 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                outputs = model(batch_x)
                val_mse += criterion(outputs, batch_y).item()
                val_mae += mae_criterion(outputs, batch_y).item()

        avg_train_loss = total_mse / len(train_loader)
        avg_val_mse = val_mse / len(val_loader)
        avg_val_mae = val_mae / len(val_loader)
        
        # Get current LRs for display
        lr_classical = optimizer.param_groups[0]['lr']
        lr_quantum = optimizer.param_groups[1]['lr']

        scheduler.step(avg_val_mse)

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train MSE: {avg_train_loss:.4f} | "
              f"Val MSE: {avg_val_mse:.4f} | "
              f"Val MAE: {avg_val_mae:.4f} | "
              f"LR(C): {lr_classical:.5f} LR(Q): {lr_quantum:.5f}")
              
    return model

In [14]:
# Assuming you have the 'preprocess_weather_data' function from the previous chat
# and your 'QuLTSF' model class defined.

# 1. Hyperparameters (from Paper)
SEQ_LEN = 336
PRED_LEN = 96
BATCH_SIZE = 32
PATCH_LEN=16
LR = 0.001
EPOCHS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Load and Preprocess Data
print("Preprocessing data...")
# Replace 'weather.csv' with your actual file path
data_dict = preprocess_weather_data('data\weather.csv', seq_len=SEQ_LEN, pred_len=PRED_LEN)

# 3. Initialize Model
# Note: Input dim is SEQ_LEN because of the reshaping
class Configs:
    def __init__(self):
        self.seq_len = 336
        self.pred_len = 96
        self.patch_len = 16
        self.num_qubits = 10
        self.num_layers = 3
        self.QML_device = "default.qubit" # Use 'lightning.qubit' if installed for speed

# 2. Instantiate the config
configs = Configs()

# 3. Instantiate the model passing the config object
model = Model(configs)

# Move to GPU if available
model = model.to(DEVICE)
model.summary()

<>:16: SyntaxWarning: invalid escape sequence '\w'
<>:16: SyntaxWarning: invalid escape sequence '\w'
C:\Users\shahj\AppData\Local\Temp\ipykernel_16100\3532098298.py:16: SyntaxWarning: invalid escape sequence '\w'
  data_dict = preprocess_weather_data('data\weather.csv', seq_len=SEQ_LEN, pred_len=PRED_LEN)


Preprocessing data...
Data Shape: (52696, 21)
Train Sequences: (36456, 336, 21)

                           Patch-QuLTSF Model Summary                           
Configuration:
  Input Sequence Length:   336
  Prediction Length:       96
  Patch Length:            16
  Stride:                  16
  Total Patches:           21
  Qubits per Patch:        4
--------------------------------------------------------------------------------
Layer / Parameter Name                   | Shape                | Params
--------------------------------------------------------------------------------
patch_qml_model.patch_encoder.weights    | [3, 4, 3]            | 36
patch_qml_model.head.weight              | [96, 84]             | 8,064
patch_qml_model.head.bias                | [96]                 | 96
patch_qml_model.skip_connection.weight   | [96, 336]            | 32,256
patch_qml_model.skip_connection.bias     | [96]                 | 96
--------------------------------------------------------

In [63]:
trained_model = train_patched_qultsf(
    model=model,
    data_dict=data_dict,     # The dictionary containing train/val/test splits
    seq_len=SEQ_LEN,
    pred_len=PRED_LEN,
    batch_size=32,           # Batch size from the paper
    epochs=10,               # Increased to 30 to help Quantum layers converge
    lr=0.0001,                # Base LR (If using improved loop, Quantum gets 0.0001)
    device=DEVICE
)

Starting training on cuda...
Classical LR: 0.001 | Quantum LR: 0.0001
Epoch 1/10 | Train MSE: 0.6367 | Val MSE: 0.5683 | Val MAE: 0.4151 | LR(C): 0.00050 LR(Q): 0.00005
Epoch 2/10 | Train MSE: 0.5528 | Val MSE: 0.5320 | Val MAE: 0.3896 | LR(C): 0.00050 LR(Q): 0.00005
Epoch 3/10 | Train MSE: 0.5368 | Val MSE: 0.5273 | Val MAE: 0.3858 | LR(C): 0.00050 LR(Q): 0.00005
Epoch 4/10 | Train MSE: 0.5312 | Val MSE: 0.5177 | Val MAE: 0.3776 | LR(C): 0.00050 LR(Q): 0.00005
Epoch 5/10 | Train MSE: 0.5236 | Val MSE: 0.5090 | Val MAE: 0.3691 | LR(C): 0.00050 LR(Q): 0.00005
Epoch 6/10 | Train MSE: 0.5145 | Val MSE: 0.5020 | Val MAE: 0.3631 | LR(C): 0.00050 LR(Q): 0.00005
Epoch 7/10 | Train MSE: 0.5025 | Val MSE: 0.4957 | Val MAE: 0.3574 | LR(C): 0.00050 LR(Q): 0.00005
Epoch 8/10 | Train MSE: 0.4986 | Val MSE: 0.4927 | Val MAE: 0.3552 | LR(C): 0.00050 LR(Q): 0.00005
Epoch 9/10 | Train MSE: 0.4957 | Val MSE: 0.4910 | Val MAE: 0.3538 | LR(C): 0.00050 LR(Q): 0.00005
Epoch 10/10 | Train MSE: 0.4934 | Val M

In [64]:
# trained_model = train_qultsf(
#     model=model,
#     data_dict=data_dict,
#     seq_len=SEQ_LEN,
#     pred_len=PRED_LEN,
#     batch_size=BATCH_SIZE,
#     epochs=EPOCHS,
#     lr=LR,
#     device=DEVICE
# )

In [ ]:
my_scaler = data_dict['scaler']

save_checkpoint(trained_model, my_scaler, "weather_qultsf.pth")

Model weights and config saved to weather_qultsf.pth
Data Scaler saved to weather_qultsf_scaler.pkl


In [ ]:
def test_and_visualize(model, scaler, data_dict, configs, device='cpu', plot_variate_idx=0):
    """
    model: The trained QuLTSF model
    data_dict: The dictionary containing data and scaler from preprocessing
    configs: Config object with seq_len, pred_len, etc.
    plot_variate_idx: Index of the variable to plot (e.g., 0 might be Temperature)
    """
    
    # 1. Prepare Data
    test_x, test_y = data_dict['test']
    
    # Create DataLoader
    test_set = WeatherDataset(test_x, test_y)
    test_loader = DataLoader(test_set, batch_size=16, shuffle=False)
    
    criterion = nn.MSELoss()
    mae_criterion = nn.L1Loss()    
    
    model.eval()
    model.to(device)
    
    total_mse = 0
    total_mae = 0
    
    # Lists to store specific samples for plotting later
    sample_inputs = []
    sample_preds = []
    sample_trues = []
    
    print("Running Inference on Test Set...")
    
    with torch.no_grad():
        for i, (batch_x, batch_y) in enumerate(test_loader):
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            # Forward Pass
            preds = model(batch_x)
            
            # Calculate Normalized Metrics (As used in the paper)
            loss = criterion(preds, batch_y)
            mae = mae_criterion(preds, batch_y)
            
            total_mse += loss.item()
            total_mae += mae.item()
            
            # Save the first batch for visualization purposes
            if i == 0:
                sample_inputs = batch_x.cpu().numpy() # [Batch, Seq_Len, M]
                sample_preds = preds.cpu().numpy()    # [Batch, Pred_Len, M]
                sample_trues = batch_y.cpu().numpy()  # [Batch, Pred_Len, M]

    # Average Metrics
    avg_mse = total_mse / len(test_loader)
    avg_mae = total_mae / len(test_loader)
    
    print("\n" + "="*40)
    print(f"TEST RESULTS (Normalized)")
    print("="*40)
    print(f"MSE: {avg_mse:.5f}")
    print(f"MAE: {avg_mae:.5f}")
    print("="*40 + "\n")

    # =========================================================
    # VISUALIZATION (Inverse Scaling)
    # =========================================================
    
    # We will pick the first sample from the saved batch to plot
    # Index 0 in the batch
    sample_idx = 0 
    
    # Extract just the specific variate we want to plot (e.g., Temperature)
    # However, to inverse_transform, we need ALL variates first because scaler expects 21 cols
    
    # 1. Prepare raw arrays for the single sample
    # Shape: [Seq_Len, Variates] -> [336, 21]
    input_seq_norm = sample_inputs[sample_idx] 
    # Shape: [Pred_Len, Variates] -> [96, 21]
    pred_seq_norm = sample_preds[sample_idx]
    true_seq_norm = sample_trues[sample_idx]
    
    # 2. Inverse Transform (Un-normalize) to get real units
    input_seq_real = scaler.inverse_transform(input_seq_norm)
    pred_seq_real = scaler.inverse_transform(pred_seq_norm)
    true_seq_real = scaler.inverse_transform(true_seq_norm)
    
    # 3. Extract the specific variate column (e.g., column 0)
    hist_data = input_seq_real[:, plot_variate_idx]
    pred_data = pred_seq_real[:, plot_variate_idx]
    true_data = true_seq_real[:, plot_variate_idx]
    
    # 4. Create X-axis coordinates
    x_hist = np.arange(0, len(hist_data))
    x_future = np.arange(len(hist_data), len(hist_data) + len(true_data))
    
    # 5. Plot
    plt.figure(figsize=(12, 6))
    
    # Plot History
    plt.plot(x_hist, hist_data, label='Historical Input', color='black', alpha=0.7)
    
    # Plot Ground Truth
    plt.plot(x_future, true_data, label='Ground Truth', color='green', marker='.')
    
    # Plot Prediction
    plt.plot(x_future, pred_data, label='QuLTSF Prediction', color='red', linestyle='--', marker='x')
    
    plt.title(f"Forecast Visualization (Variate Index: {plot_variate_idx})")
    plt.xlabel("Time Steps")
    plt.ylabel("Value (Original Units)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# 1. Define your Configs (Assuming you used the class approach)
class Configs:
    def __init__(self):
        self.seq_len = 336
        self.pred_len = 96
        self.num_qubits = 10
        self.num_layers = 3
        self.QML_device = 'lightning.qubit'

configs = Configs()

loaded_model, loaded_scaler = load_checkpoint("checkpoints\qltsf-patch.pth", device=DEVICE)

# 2. Run the Test
# 'trained_model' comes from your training loop
# 'data_dict' comes from your preprocessing step
# 'plot_variate_idx=20' (In Weather dataset, index 20 is usually 'OT' - Oil Temperature/Target)

test_and_visualize(
    model=loaded_model,
    scaler=loaded_scaler,
    data_dict=data_dict, 
    configs=configs, 
    device=DEVICE,
    plot_variate_idx=7 # Try 0 or 20 depending on which variable you want to see
)